In [ ]:
!pip install torch


In [ ]:
import os
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch  # 导入 PyTorch

## 选择基础模型

DeepSeek 团队选择 DeepSeek-V3 作为基础模型来创建 R1 Zero 和 R1，但它非常庞大，高达 685 GB 💀，这显然超出了我们的承受范围。

为了简单起见，我们将使用一个更小的基础模型 [Qwen/Qwen2.5–0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)（大小为 0.9 GB）。如果您拥有更高的 GPU 显存，甚至可以加载未量化的 LLM，那么您可以选择更大的模型，例如 [Qwen/Qwen2.5–7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)。

让我们来看看基础模型的一些规格：

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "data/Qwen-GRPO-training"  # 为了保存我们训练好的模型

# 如果输出目录不存在，则创建它
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 使用聊天模板初始化标记器
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right"
)

# 如果未设置，则设置 pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulary size: {len(tokenizer)}")
print(f"Model max length: {tokenizer.model_max_length}")

print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

这些是有关模型的一些基本信息，请查看我们的基础模型的参数总数。

In [ ]:
# 初始化基础模型
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  trust_remote_code=True,
  torch_dtype=torch.bfloat16
)

print(f"Model parameters: {model.num_parameters():,}")

接近 0.5B 的参数，让我们从中打印一个简单的响应，然后进入下一步。

In [ ]:
# 检查 CUDA 可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 将模型移动到适当的设备
model.to(device)

# 测试基本推理
def test_model_inference(user_input: str):
    """Test basic model inference with the loaded model and tokenizer."""
    messages = [
        {"role": "system", "content": "You are Qwen, a helpful assistant."},
        {"role": "user", "content": user_input}
    ]

    # 应用聊天模板
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 标记化并生成
    inputs = tokenizer(text, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# 测试模型
test_input = "how are you?"
response = test_model_inference(test_input)
print(f"Test Input: {test_input}")
print(f"Model Response: {response}")



因此，这个小模型的输出非常可靠，并且肯定适合我们的 DeepSeek 相似模型训练。